In [12]:
# import libraries
import re
import string
import heapq
import numpy as np
import pandas as pd
from tqdm import tqdm
from urllib.request import urlopen

# if you need to plot anything
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
import matplotlib.pyplot as plt

# Stopword dictionary
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
from nltk import download, wordnet
nltk.download('omw-1.4')
download('punkt')
download('stopwords')
download("wordnet")

# For stemming
from nltk.stem.porter import PorterStemmer
from nltk.stem import WordNetLemmatizer
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()


from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
# cv = CountVectorizer()
# tfidf = TfidfVectorizer()

# let's import ML algorithms for regression or classification
# from sklearn.svm import SVC
from cuml import SVR, SVC, LinearSVR
from sklearn.linear_model import LogisticRegression, ElasticNet
from sklearn.preprocessing import StandardScaler, MinMaxScaler, MaxAbsScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split as tts, KFold, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score as ac, confusion_matrix as cm, ConfusionMatrixDisplay as CMD

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, TensorDataset
# import torch.nn.functional as F

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/dvasiliu/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/dvasiliu/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt to /home/dvasiliu/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/dvasiliu/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/dvasiliu/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [2]:
# obtain the data
wine_data = pd.read_csv('https://github.com/dvasiliu/AML/blob/main/Data%20Sets/winemagdata130kv2.csv?raw=true',quoting=2)
wines = wine_data[["description","points"]]
# wines_subset = wines.sample(1000,random_state=1234).reset_index(drop=True)

In [3]:
wines

,description,points
0,"Aromas include tropical fruit, broom, brimston...",87.0
1,"This is ripe and fruity, a wine that is smooth...",87.0
2,"Tart and snappy, the flavors of lime flesh and...",87.0
3,"Pineapple rind, lemon pith and orange blossom ...",87.0
4,"Much like the regular bottling from 2012, this...",87.0
...,...,...
129966,Notes of honeysuckle and cantaloupe sweeten th...,90.0
129967,Citation is given as much as a decade of bottl...,90.0
129968,Well-drained gravel soil gives this wine its c...,90.0
129969,"A dry style of Pinot Gris, this is crisp with ...",90.0


In [3]:
# the idea here is to preprocess before any embedding
# our preprocessing is minimal: lower all cases, remove punctuation, and also remove the stopwords.
# -> we really want to reduce the size of the corpus
def fast_preprocess(texts, lemmatize=False):
    
    # Fast text preprocessing function that handles:
    # - Lowercasing
    # - Punctuation removal
    # - Stopword removal
    # - Lemmatization (optional)
    
    # Arguments:
    #     texts: List of strings or single string to process
    #     stop_words: Set of stopwords (if None, uses NLTK English stopwords)
    #     lemmatize: Whether to perform lemmatization
    
    # Returns:
    #     Processed texts (list or string, matching input type)
    
    # Handle single string input
    single_input = isinstance(texts, str)
    if single_input:
        texts = [texts]
    
    
    
    stop_words = set(stopwords.words('english'))
    
    # Create translator for punctuation removal (faster than regex for this task)
    translator = str.maketrans('', '', string.punctuation)
    
    # Initialize lemmatizer if needed
    lemmatizer = WordNetLemmatizer() if lemmatize else None
    
    processed_texts = []
    for text in texts:
        # Lowercase
        text = text.lower()
        
        # Remove punctuation
        text = text.translate(translator)
        
        # Tokenize (needed for both stopword removal and lemmatization)
        tokens = text.split()
        
        # Remove stopwords
        tokens = [token for token in tokens if token not in stop_words]
        
        # Lemmatize if requested
        if lemmatize:
            tokens = [lemmatizer.lemmatize(token) for token in tokens]
        
        # Rejoin into text
        processed_texts.append(' '.join(tokens))
    
    # Return in same format as input
    return processed_texts[0] if single_input else processed_texts

In [4]:
prep_reviews=fast_preprocess(wines.description,lemmatize=False)

Critical Thinking: How many distinct tokens would be needed for a good quality sentiment analysis?
One possible answer: we can treat this as a hyperparameter -> this could be optimized with a grid search algoriuthm such as simulated annealing.

In [5]:
# to control the size of the corpus we can use max_features for the TF-IDF vectorizer
n_features = 2000
embedding = TfidfVectorizer(max_features=n_features,ngram_range=(1, 3),norm='l2')

In [ ]:
# this needs to be used with the logic of dealing with Train/Test data
# xtrain = embedding.fit_transform(xtrain); xtest = embedding.transform(xtest)

In [6]:
# we want to conduct the sentiment analysis and measure the predictive power with k-fold cv
kf = KFold(n_splits=5, shuffle=True)
x = prep_reviews
y = wines.points

In [23]:
# the question: how to slice a list of lists (reach review is a list of ACSCII characters)
np.array(x)[[0,1]]

array(['aromas include tropical fruit broom brimstone dried herb palate isnt overly expressive offering unripened apple citrus dried sage alongside brisk acidity',
       'ripe fruity wine smooth still structured firm tannins filled juicy red berry fruits freshened acidity already drinkable although certainly better 2016'],
      dtype='<U614')

In [8]:
# we can code ourt own loop for cross-validation
mse = []

for idxtrain, idxtest in tqdm(kf.split(x)):
    
    xtrain = embedding.fit_transform(np.array(x)[idxtrain])
    ytrain = y[idxtrain]

    xtest = embedding.transform(np.array(x)[idxtest])
    ytest = y[idxtest]

    model = SVR(kernel='linear', epsilon=5, C=10)
    model.fit(xtrain,ytrain)
    mse.append(np.mean((ytest-model.predict(xtest))**2))

print(f'The External MSE validation is :{np.mean(mse):.4f}')

0it [00:00, ?it/s]

[2025-07-30 15:29:27.339] [CUML] [warning] SVR with the linear kernel can be much faster using the specialized solver provided by LinearSVR. Consider switching to LinearSVR if tranining takes too long.


5it [01:09, 13.84s/it]

The External MSE validation is :4.1993


## Example with a Neural Netowrk

In [7]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [10]:
n_features

2000

### Example with Dense representations of the torch tensors -> very , very slow to train!

In [19]:

x_train, x_test, y_train, y_test = tts(x, y.values, test_size=0.2)

x_train = embedding.fit_transform(x_train)
x_test = embedding.transform(x_test)

# we have to put our data in sparse torch arrays
def scipy_sparse_to_torch_sparse(sparse_matrix, device='cuda'):
    """Convert scipy sparse matrix to PyTorch sparse tensor"""
    sparse_matrix = sparse_matrix.tocoo()  # Convert to COO format
    indices = torch.from_numpy(np.vstack((sparse_matrix.row, sparse_matrix.col))).long()
    values = torch.from_numpy(sparse_matrix.data).float()
    shape = sparse_matrix.shape
    
    # Create sparse tensor and move to device
    sparse_tensor = torch.sparse_coo_tensor(indices, values, shape)
    return sparse_tensor.to(device)

# Convert to sparse tensors and move to GPU
x_train = scipy_sparse_to_torch_sparse(x_train, device)
x_test = scipy_sparse_to_torch_sparse(x_test, device)
y_train = torch.tensor(y_train, dtype=torch.float32).view(-1, 1).to(device)
y_test = torch.tensor(y_test, dtype=torch.float32).view(-1, 1).to(device)


# Define a simple neural network for regression
class regressionNN(nn.Module):
    def __init__(self,n_features):
        super(regressionNN, self).__init__()
        self.layer1 = nn.Linear(n_features, 500)
        self.layer2 = nn.Linear(500, 200)
        self.layer3 = nn.Linear(200, 1)
        # Define PReLU activations as instance variables
        self.prelu1 = nn.PReLU()
        self.prelu2 = nn.PReLU()

    def forward(self, x):
        # Convert sparse tensor to dense for linear layers
        if x.is_sparse:
            x = x.to_dense()
        x = self.prelu1(self.layer1(x))
        x = self.prelu2(self.layer2(x))
        x = self.layer3(x)
        return x

# Instantiate the model, define loss function and optimizer
model = regressionNN(n_features).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)


train_dataset = TensorDataset(x_train.to_dense(), y_train)
test_dataset = TensorDataset(x_test.to_dense(), y_test)

# Use smaller batch sizes
batch_size = 16  # Adjust this based on your GPU memory
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Modified training loop
n_epochs = 10
for epoch in range(n_epochs):
    model.train()
    total_loss = 0
    
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    if (epoch+1) % 10 == 0:
        avg_loss = total_loss / len(train_loader)
        print(f'Epoch [{epoch+1}/{n_epochs}], Average Loss: {avg_loss:.4f}')

# Evaluation with batches
model.eval()
total_test_loss = 0
with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        predictions = model(batch_x)
        test_loss = criterion(predictions, batch_y)
        total_test_loss += test_loss.item()

avg_test_loss = total_test_loss / len(test_loader)
print(f'Average Test Loss: {avg_test_loss:.4f}')


Epoch [10/10], Average Loss: 136.5495
Average Test Loss: 5.5590


## Example of training the Neural Network with sparse representations of the data

In [32]:
# Define a simple neural network for regression
class SimpleNN(nn.Module):
    def __init__(self, n_features):
        super(SimpleNN, self).__init__()
        self.layer1 = nn.Linear(n_features, 500)
        self.layer2 = nn.Linear(500, 500)
        self.layer3 = nn.Linear(500, 1)

    def forward(self, x):
        x = torch.relu(self.layer1(x))
        x = torch.relu(self.layer2(x))
        x = self.layer3(x)
        return x

# Instantiate the model, define loss function and optimizer
model = SimpleNN(n_features).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Training the model
n_epochs = 200
for epoch in range(n_epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(x_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()

    if (epoch+1) % 25 == 0:
        print(f'Epoch [{epoch+1}/{n_epochs}], Loss: {loss.item():.4f}')

# Evaluating the model
model.eval()
with torch.no_grad():
    predictions = model(x_test)
    test_loss = criterion(predictions, y_test)
    print(f'Test Loss: {test_loss.item():.4f}')

Epoch [25/200], Loss: 478.1164
Epoch [50/200], Loss: 45.5156
Epoch [75/200], Loss: 40.6184
Epoch [100/200], Loss: 33.0541
Epoch [125/200], Loss: 17.4172
Epoch [150/200], Loss: 4.6267
Epoch [175/200], Loss: 3.2315
Epoch [200/200], Loss: 3.0949
Test Loss: 3.3836


## Treat the Sentiment Analysis Problem as Classification

In [ ]:
pts = wine_subset.points
y = pts.copy().values
y[pts>=92] = 4
y[(pts>=90) & (pts<=91)] = 3
y[(pts>=88) & (pts<=89)] = 2
y[(pts>=86) & (pts<=87)] = 1
y[(pts>=80) & (pts<=85)] = 0